# 05 - O que a IA viu: evidencia estruturada

Audita a disponibilidade dos sinais pedidos na reuniao e formaliza um contrato que separa observacao, decisao e abstencao. Texto livre nunca vira verdade-terreno.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd
ROOT = Path.cwd(); NB_DIR = ROOT / 'notebooks' if (ROOT / 'notebooks').is_dir() else ROOT
sys.path.insert(0, str(NB_DIR))
from produtividade_30d import carregar_fontes, preparar_dataset, dividir_por_dia, normalizar_serie
OUT = Path(os.environ.get('KV_30D_OUTPUT_DIR', NB_DIR / 'outputs' / 'productivity_30d')); OUT.mkdir(parents=True, exist_ok=True)
eventos, catalogo, _ = carregar_fontes(); proxy, _ = preparar_dataset(eventos, catalogo); split = dividir_por_dia(proxy)
campos = ['descricao_bruta','trabalho','orientacao','maos_maquina','modo_operacao','movimento_maquina','cena_maquina','cena_imovel','produtividade_motivo','bbox_stats']
linhas = []
for parte, df in [('treino', split.treino), ('calibracao', split.calibracao), ('teste_interno', split.teste_interno)]:
    for campo in campos:
        cobertura = 0.0 if campo not in df else ((df[campo].notna()) & (df[campo].astype(str).str.strip() != '')).mean()
        linhas.append({'parte': parte, 'campo': campo, 'cobertura': float(cobertura), 'n': len(df)})
auditoria = pd.DataFrame(linhas)
schema = {
 'operador_estado': ['identificado','ausente','incerto'],
 'acao_observada': 'texto curto, observacional, sem julgamento',
 'trabalho': [True, False, None],
 'motivo': ['maos_no_torno','voltado_para_torno','uso_celular','sem_atividade','conversa','sem_leitura'],
 'maos_maquina': [True, False, None],
 'maquina_estado': ['operacao_manual','ciclo_automatico','parada','incerto'],
 'ponte_evidencias': ['controle','gancho','carga_suspensa','movimento_coordenado','confirmacao_cam2'],
 'conversa_estado': ['ausente','identificada','incerta'],
 'interlocutor_tipo': ['gestor_cinza','outra_pessoa','incerto'],
 'confianca': '0..1',
 'regra': 'false exige motivo especifico + persistencia; conflito vira null'
}
auditoria.to_csv(OUT / 'cobertura_evidencia_estruturada.csv', index=False)
(OUT / 'schema_evidencia_produtividade.json').write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding='utf-8')
display(auditoria.pivot(index='campo', columns='parte', values='cobertura'))
print(json.dumps(schema, indent=2, ensure_ascii=False))
print('produtividade_motivo no export:', 'produtividade_motivo' in eventos.columns)